In [ ]:
# 1. Install / upgrade the Hugging Face datasets library
!pip install -q -U datasets

# 2. Mount Google Drive so files survive Colab runtime resets
from google.colab import drive
drive.mount('/content/drive')

# 3. Create a folder on Drive for all project files
import os
DATA_DIR = "/content/drive/MyDrive/code_completion"
os.makedirs(DATA_DIR, exist_ok=True)

# 4. Record the Python version (the tokenizer behaves differently on 3.12+)
import sys
print("Python:", sys.version.split()[0])
print("Saving files to:", DATA_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 17.2 MB/s eta 0:00:00
Mounted at /content/drive
Python: 3.13.16
Saving files to: /content/drive/MyDrive/code_completion


In [ ]:
from collections import Counter
from datasets import load_dataset

# 1. Load the dataset (split="train" returns the data directly)
ds = load_dataset("Nan-Do/code-search-net-python", split="train")

# 2. Read the partition label of every function
partitions = ["valid" if p == "validation" else p for p in ds["partition"]]
print("partition values:", Counter(partitions))

# 3. Group the raw code strings by partition
codes = {"train": [], "valid": [], "test": []}
for code, part in zip(ds["code"], partitions):
    codes[part].append(code)

# 4. Check the sizes
print({name: len(items) for name, items in codes.items()})

README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

data/train-00000-of-00004-ee77a7de79eb2a(…): reconstructing file:   0%|          |  0.00B /  155MB            

data/train-00000-of-00004-ee77a7de79eb2a(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-648b3bede2edf6(…): reconstructing file:   0%|          |  0.00B /  139MB            

data/train-00001-of-00004-648b3bede2edf6(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-1dfd72b171e6b2(…): reconstructing file:   0%|          |  0.00B /  153MB            

data/train-00002-of-00004-1dfd72b171e6b2(…): downloading bytes:           |  0.00B            

data/train-00003-of-00004-184ab6d0e3c690(…): reconstructing file:   0%|          |  0.00B /  151MB            

data/train-00003-of-00004-184ab6d0e3c690(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/455243 [00:00<?, ? examples/s]

partition values: Counter({'train': 410175, 'valid': 22977, 'test': 22091})
{'train': 410175, 'valid': 22977, 'test': 22091}


In [ ]:
import io, random, tokenize

SKIP = {tokenize.NEWLINE, tokenize.NL, tokenize.INDENT, tokenize.DEDENT,
        tokenize.ENDMARKER, tokenize.COMMENT}
FS_START = getattr(tokenize, "FSTRING_START", None)   # exists on Python 3.12+
FS_END   = getattr(tokenize, "FSTRING_END", None)

def line_starts(src):
    """Character offset where each line begins."""
    starts = [0]
    for line in io.StringIO(src).readlines():
        starts.append(starts[-1] + len(line))
    return starts

def clean_source(src):
    """Remove comments and docstrings, keep the original spacing."""
    try:
        toks = list(tokenize.generate_tokens(io.StringIO(src).readline))
    except Exception:
        return None
    starts, spans, prev = line_starts(src), [], tokenize.INDENT
    for k, tok in enumerate(toks):
        nxt = toks[k + 1].type if k + 1 < len(toks) else None
        is_doc = (tok.type == tokenize.STRING and nxt == tokenize.NEWLINE
                  and prev in (tokenize.INDENT, tokenize.NEWLINE, tokenize.DEDENT))
        if tok.type == tokenize.COMMENT or is_doc:
            spans.append((starts[tok.start[0] - 1] + tok.start[1],
                          starts[tok.end[0] - 1] + tok.end[1]))
        if tok.type not in (tokenize.COMMENT, tokenize.NL):
            prev = tok.type
    for a, b in sorted(spans, reverse=True):
        src = src[:a] + src[b:]
    return "\n".join(l.rstrip() for l in src.split("\n") if l.strip())

def make_pairs(code, k, rng, ctx_chars=1000, max_target_chars=150):
    """Return up to k (prefix, target) pairs from one function."""
    src = clean_source(code)
    if not src:
        return []
    starts, cands, depth = line_starts(src), [], 0
    try:
        for tok in tokenize.generate_tokens(io.StringIO(src).readline):
            if tok.type == FS_START:          # never cut inside an f-string
                depth += 1
                continue
            if tok.type == FS_END:
                depth -= 1
            if depth > 0 or tok.type in SKIP:
                continue
            cut = starts[tok.end[0] - 1] + tok.end[1]     # cut right after a token
            nl = src.find("\n", cut)
            target = src[cut: len(src) if nl == -1 else nl]
            if len(target.strip()) >= 4 and len(target) <= max_target_chars:
                cands.append((cut, target))
    except Exception:
        return []
    rng.shuffle(cands)
    return [{"prefix": src[max(0, cut - ctx_chars):cut], "target": t}
            for cut, t in cands[:k]]

In [ ]:
import json
from tqdm.auto import tqdm

def key_of(p):
    return (p["prefix"][-200:], p["target"])

def build(codes_list, n_pairs, per_func, seed, exclude=frozenset()):
    rng = random.Random(seed)
    order = list(range(len(codes_list)))
    rng.shuffle(order)
    pairs, seen = [], set()
    bar = tqdm(total=n_pairs)
    for i in order:
        for p in make_pairs(codes_list[i], per_func, rng):
            k = key_of(p)
            if k in seen or k in exclude:
                continue
            seen.add(k)
            pairs.append(p)
            bar.update(1)
        if len(pairs) >= n_pairs:
            break
    bar.close()
    return pairs[:n_pairs]

def save_jsonl(pairs, name):
    path = os.path.join(DATA_DIR, name)
    with open(path, "w") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")
    print(path, len(pairs))

test_pairs  = build(codes["test"],  5_000, 1, seed=2)
valid_pairs = build(codes["valid"], 5_000, 1, seed=1)
held_out    = {key_of(p) for p in test_pairs + valid_pairs}
train_pairs = build(codes["train"], 100_000, 2, seed=0, exclude=held_out)

save_jsonl(test_pairs,  "test.jsonl")
save_jsonl(valid_pairs, "valid.jsonl")
save_jsonl(train_pairs, "train.jsonl")

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/100000 [00:00<?, ?it/s]

/content/drive/MyDrive/code_completion/test.jsonl 5000
/content/drive/MyDrive/code_completion/valid.jsonl 5000
/content/drive/MyDrive/code_completion/train.jsonl 100000


In [ ]:
import statistics

for name, pairs in [("train", train_pairs), ("valid", valid_pairs), ("test", test_pairs)]:
    tl = [len(p["target"]) for p in pairs]
    print(f"{name}: {len(pairs)} pairs | target chars: mean {statistics.mean(tl):.0f}, "
          f"median {statistics.median(tl):.0f}, max {max(tl)}")

assert all("\n" not in p["target"] for p in train_pairs + valid_pairs + test_pairs)
print("train/test overlap:",
      len({key_of(p) for p in train_pairs} & {key_of(p) for p in test_pairs}))

for p in random.Random(0).sample(test_pairs, 5):
    print("-" * 60)
    print(p["prefix"][-120:] + "⟦CURSOR⟧" + p["target"])

train: 100000 pairs | target chars: mean 23, median 19, max 150
valid: 5000 pairs | target chars: mean 23, median 19, max 145
test: 5000 pairs | target chars: mean 23, median 18, max 128
train/test overlap: 0
------------------------------------------------------------
 today = timezone.utcnow().replace(
        hour=hour,
        minute=minute,
        second=second,
        microsecond⟦CURSOR⟧=microsecond)
------------------------------------------------------------
def⟦CURSOR⟧ file_read(filename):
------------------------------------------------------------
rompt(self, cursor):
        if self._continuation_prompt_html is None:
            self._insert_plain_text(cursor, self⟦CURSOR⟧._continuation_prompt)
------------------------------------------------------------
ag=1, max_size=1,
                   ref=None,
                   detrend=False, center=True,
                   feature⟦CURSOR⟧=None, aggregate=None,
------------------------------------------------------------
f.nd_json